In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# 1. Configuration & Hyperparameters
DATA_DIR = r'C:\ai model for krishi\data' # Ensure this matches your shortened path
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

# Auto-detect RTX GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Data Augmentation and Normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 3. Load Dataset using ImageFolder
image_datasets = {
    x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=0, pin_memory=True)
    for x in ['train', 'val']
}

class_names = image_datasets['train'].classes
print(f"Classes detected: {class_names}")

# 4. Initialize Pre-trained Model (Transfer Learning)
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Freeze early layers to retain generic image features
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification head to match the dynamic folder count
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

# 5. Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

# 6. Training Loop with Mixed Precision (AMP)
scaler = torch.amp.GradScaler('cuda')

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # Forward pass
            with torch.set_grad_enabled(phase == 'train'):
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                # Backward pass + optimize
                if phase == 'train':
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])
        
        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

# Save the trained model weights
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_mapping': class_names 
}
torch.save(checkpoint, 'tomato_multi_disease_model.pth')
print(f"\nTraining complete. Saved model and class mapping: {class_names}")


Training on device: cuda
Classes detected: ['Bacterial_spot', 'Early_blight', 'Late_blight', 'Leaf_Mold', 'Septoria_leaf_spot', 'Spider_mites Two-spotted_spider_mite', 'Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus', 'Tomato_mosaic_virus', 'healthy', 'powdery_mildew']

Epoch 1/10
----------
Train Loss: 1.2754 Acc: 0.5851
Val Loss: 0.8363 Acc: 0.7363

Epoch 2/10
----------
Train Loss: 0.9667 Acc: 0.6794
Val Loss: 0.7338 Acc: 0.7637

Epoch 3/10
----------
Train Loss: 0.8988 Acc: 0.7036
Val Loss: 0.6895 Acc: 0.7772

Epoch 4/10
----------
Train Loss: 0.8696 Acc: 0.7084
Val Loss: 0.7201 Acc: 0.7622

Epoch 5/10
----------
Train Loss: 0.8606 Acc: 0.7083
Val Loss: 0.6709 Acc: 0.7769

Epoch 6/10
----------
Train Loss: 0.8423 Acc: 0.7194
Val Loss: 0.7007 Acc: 0.7675

Epoch 7/10
----------
Train Loss: 0.8456 Acc: 0.7151
Val Loss: 0.7466 Acc: 0.7443

Epoch 8/10
----------
Train Loss: 0.8313 Acc: 0.7175
Val Loss: 0.6539 Acc: 0.7839

Epoch 9/10
----------
Train Loss: 0.8347 Acc: 0.7179
Val Loss: 0.67